In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../')
from utils.MultiLabelPredictor import MultilabelPredictor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

/home/massimo/Documents/stage/signature_inference/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def best_for_each(r2s,mses):
    best_r2_idxs = np.argmax(r2s, axis=0)
    best_mse_idxs = np.argmin(mses, axis=0)
    best_r2_idx = Counter(best_r2_idxs).most_common(1)[0][0] 
    best_mse_idx = Counter(best_mse_idxs).most_common(1)[0][0] 
    return best_r2_idx == best_mse_idx, best_mse_idx 

In [3]:
class PredictExposure:
    def __init__(self, model):
        self.model = model 

    def fit(self, mutation_count):

        mutation_count_bin = pd.read_csv('../simulations/ground_truth/bin_exposures.csv').iloc[:,1:].astype(int).values  # N x 29
        signature_exposure = pd.read_csv('../simulations/ground_truth/exposures.csv').iloc[:,1:].values  # N x 29

        X_train, X_test, y_bin_train, y_bin_test, y_exp_train, y_exp_test = train_test_split(
            mutation_count, mutation_count_bin, signature_exposure, train_size=0.8,
        )
        regressors = []
        r2_scores = []
        mse_scores = []
        predictions = []

        for i in range(y_bin_train.shape[1]):  # ciclo sulle 29 firme           # Fa predizioni anche su segnature che nono dovrebbero esserci capire come mais
            # Filtra campioni con firma attiva in train
            idx_train = y_bin_train[:, i] == 1
            pdf(idx_train)
            X_train_i = X_train[idx_train]
            pdf(X_train_i)
            y_train_i = y_exp_train[idx_train, i]
            pdf(y_train_i)

            # Filtra campioni con firma attiva in test
            idx_test = y_bin_test[:, i] == 1
            X_test_i = X_test[idx_test]
            y_test_i = y_exp_test[idx_test, i]

            if len(y_train_i) == 0 or len(y_test_i) == 0:
                regressors.append(None)
                r2_scores.append(np.nan)
                mse_scores.append(np.nan)
                continue

            self.model.fit(X_train_i, y_train_i)
            regressors.append(self.model)
            y_pred_i = self.predict(X_test_i)
            predictions.append(y_pred_i)
            r2_scores.append(r2_score(y_test_i, y_pred_i))
            mse_scores.append(mean_squared_error(y_test_i, y_pred_i))
            
        # # Output risultati
        # for i in range(len(regressors)):
        #     print(f"Signature_{i+1}: R2 = {r2_scores[i]:.3f}, MSE = {mse_scores[i]:.3e}")
        return r2_scores, mse_scores, predictions

    def predict(self, X_test_i):
            y_pred_i = self.model.predict(X_test_i)
            return y_pred_i

In [4]:
class PredictExposure:
    def __init__(self, model):
        self.model = model
        self.regressors = []
        self.mutation_count_bin = pd.read_csv('../simulations/ground_truth/bin_exposures.csv').iloc[:,1:].astype(int).values  # N x 29
        self.signature_exposure = pd.read_csv('../simulations/ground_truth/exposures.csv').iloc[:,1:].values  # N x 29


    def fit(self, train_data):
        for i in range(self.mutation_count_bin.shape[1]):  # ciclo sulle 29 firme
            # Filtra campioni con firma attiva in train
            idx_train = self.mutation_count_bin[:, i] == 1
            X_train_i = train_data[idx_train]
            y_train_i = self.signature_exposure[idx_train, i]

            self.model.fit(X_train_i, y_train_i)
            self.regressors.append(self.model)
            
    def evaluate(self, evaluate):
        X_train, X_test, y_bin_train, y_bin_test, y_exp_train, y_exp_test = train_test_split(
            evaluate, self.mutation_count_bin, self.signature_exposure, train_size=0.8,
        )
        r2_scores = []
        mse_scores = []
        predictions = []

        for i in range(y_bin_train.shape[1]):  # ciclo sulle 29 firme
            # Filtra campioni con firma attiva in train
            idx_train = y_bin_train[:, i] == 1
            X_train_i = X_train[idx_train]
            y_train_i = y_exp_train[idx_train, i]

            # Filtra campioni con firma attiva in test
            idx_test = y_bin_test[:, i] == 1
            X_test_i = X_test[idx_test]
            y_test_i = y_exp_test[idx_test, i]

            if len(y_train_i) == 0 or len(y_test_i) == 0:
                self.regressors.append(None)
                r2_scores.append(np.nan)
                mse_scores.append(np.nan)
                continue

            regressor = self.model.fit(X_train_i, y_train_i)
            self.regressors.append(regressor)
            y_pred_i = regressor.predict(X_test_i)
            predictions.append(y_pred_i)
            r2_scores.append(r2_score(y_test_i, y_pred_i))
            mse_scores.append(mean_squared_error(y_test_i, y_pred_i))
            
        # # Output risultati
        # for i in range(len(regressors)):
        #     print(f"Signature_{i+1}: R2 = {r2_scores[i]:.3f}, MSE = {mse_scores[i]:.3e}")
        return r2_scores, mse_scores, predictions

    def predict(self, test_data):
        y_pred_i = []
        for i in range(self.mutation_count_bin.shape[1]):
            idx_test = self.mutation_count_bin[:, i] == 1
            X_test_i = test_data[idx_test]
            y_test_i = self.signature_exposure[idx_test, i]
            y_pred_i.append(self.regressors[i].predict(X_test_i))
        return y_pred_i
            

In [5]:
predictor = MultilabelPredictor.load('../models/saved/Predictor-0.03')
pred_mutation_count = pd.read_csv('../simulations/data2/run_1/trinucleotides_counts_sampling_0.03.csv').iloc[:,1:]
prediction = predictor.predict(pred_mutation_count)

This means that the predictor was fit in an AutoGluon version `<=0.3.1`.


Predicting with TabularPredictor for label: S1 (SBS1 - 0.99) ...


FileNotFoundError: [Errno 2] No such file or directory: '/home/baderlab/mzarant/SignatureInference/models/saved/Predictor-0.03/Predictor_S1 (SBS1 - 0.99)/predictor.pkl'

### Train's Data

In [ ]:
seed = np.random.randint(1,100000)
np.random.seed(seed=seed)
# Loading the data
signature_prob_distribution = pd.read_csv('../simulations/ground_truth/signatures.csv').iloc[:,1:].values
mutation_count = pd.read_csv('../simulations/data/run_1/trinucleotides_counts_sampling_0.03.csv').iloc[:,1:].values  # N x 96
mutation_count_bin = pd.read_csv('../simulations/ground_truth/bin_exposures.csv').iloc[:,1:].astype(int).values  # N x 29
tissues = pd.read_csv('../simulations/ground_truth/tumor_site.csv').iloc[:,1:-1].values

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
encoded_labels = pd.DataFrame(encoder.fit_transform(tissues))

In [ ]:
with_bin_mutation_count = np.hstack([mutation_count @ signature_prob_distribution.T, mutation_count_bin])
pe = PredictExposure(LinearRegression())
r2, mse, pred = pe.fit(with_bin_mutation_count)
df = pd.DataFrame(pred)
df
# print(f'Mean: {np.mean(r2, axis=0)}, Median {np.median(r2, axis=0)}, Max/Min: {np.max(r2, axis=0)}/{np.min(r2, axis=0)}')  #83637

In [ ]:
pred_with_bin_mutation_count = np.hstack([mutation_count @ signature_prob_distribution.T, prediction])
pred = pe.predict(pred_with_bin_mutation_count)
df = pd.DataFrame(pred)
df
# print(f'Mean: {np.mean(pred_r2, axis=0)}, Median {np.median(pred_r2, axis=0)}, Max/Min: {np.max(pred_r2, axis=0)}/{np.min(pred_r2, axis=0)}')  #83637

In [ ]:
seed

In [ ]:
scaler = StandardScaler()
mutation_scaled = scaler.fit_transform(mutation_count)
with_bin_mutation_count = np.hstack([mutation_count @ signature_prob_distribution.T, mutation_count_bin])
with_bin_sites_mutation_count = np.hstack([mutation_count @ signature_prob_distribution.T, mutation_count_bin,encoded_labels])

models = [LinearRegression(),
          LinearRegression(),
          LinearRegression(),
          LinearRegression(),       # BEST
          Ridge(alpha=0.1),
          LinearRegression()
          ]
fit_inputs = [mutation_count,                                   # Simple count of mutation per sample
              mutation_count @ signature_prob_distribution.T,   # Count of mutation per sample multiplied by the mutation distribution
              mutation_scaled @ signature_prob_distribution.T,  # Count of mutation per sample normalized multiplied by the mutation distribution
              with_bin_mutation_count, # BEST                   # Count multiplied by probability distribution concat with the True/Fasle vector
              mutation_count @ signature_prob_distribution.T,   # Count of mutation per sample multiplied by the mutation distribution
              with_bin_sites_mutation_count
              ]

bests = []
best_r2s = []
best_mses = []
i = 0
while i in range(50):
    i+=1
    r2s = []
    mses = []
    print(f'fitting {i}')
    for model, fit_input in zip(models, fit_inputs):
        pe = PredictExposure(model)
        fit_r2, fit_mse = pe.fit(fit_input)
        r2s.append(fit_r2),
        mses.append(fit_mse)
    match, best = best_for_each(r2s,mses)
    if match:
        print(f'Match on {best}')
        print(r2s[best], mses[best])
        print(best)
        bests.append(best)
        best_mses.append(mses[best])
        best_r2s.append(r2s[best])

In [ ]:
Counter(bests).most_common(1)[0][0] 


In [ ]:
df = pd.concat([pd.DataFrame(bests, columns=['best']), pd.DataFrame(best_r2s)], axis=1)
df[df['best'] == 5].iloc[:, 1:]